# Lung Segmentation — Offline Augmentation (Kaggle CPU)
Generates ~40k augmented image/mask pairs and saves them directly as a raw folder
output (no zipping) for use as a Kaggle Dataset in Colab training.

## Cell 1 — Clean /kaggle/working/ before anything else

In [ ]:
import os, shutil, glob

working_dir = '/kaggle/working'
deleted_files = 0
deleted_dirs  = 0

for item in glob.glob(os.path.join(working_dir, '*')):
    if os.path.isfile(item) or os.path.islink(item):
        os.remove(item)
        deleted_files += 1
    elif os.path.isdir(item):
        shutil.rmtree(item)
        deleted_dirs += 1

free_gb = shutil.disk_usage(working_dir).free / 1e9
print(f'Cleared /kaggle/working/')
print(f'  Deleted files : {deleted_files}')
print(f'  Deleted dirs  : {deleted_dirs}')
print(f'  Free space    : {free_gb:.1f} GB')

## Cell 2 — Install & check Albumentations version

In [ ]:
!pip install -q --upgrade albumentations

import albumentations as A
print(f'Albumentations version: {A.__version__}')

## Cell 3 — Imports & Config

In [ ]:
import os, glob, time, json, shutil
import cv2
import numpy as np
from PIL import Image
from multiprocessing import Pool, cpu_count

CXRAY_ROOT   ='/kaggle/input/datasets/felipemeganha/chest-xray-masks-and-labels-images'
WORK_DIR     = '/kaggle/working'
AUG_IMG_DIR  = os.path.join(WORK_DIR, 'aug_cxray', 'images')
AUG_MASK_DIR = os.path.join(WORK_DIR, 'aug_cxray', 'masks')

IMG_SIZE       = 256
TARGET_SAMPLES = 40000
N_WORKERS      = max(1, cpu_count() - 1)

os.makedirs(AUG_IMG_DIR,  exist_ok=True)
os.makedirs(AUG_MASK_DIR, exist_ok=True)

print(f'CPUs available : {cpu_count()}')
print(f'Workers        : {N_WORKERS}')
print(f'Free space     : {shutil.disk_usage(WORK_DIR).free / 1e9:.1f} GB')

## Cell 4 — Inspect dataset folder structure

In [ ]:
for root, dirs, files in os.walk(CXRAY_ROOT):
    level = root.replace(CXRAY_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        subindent = '  ' * (level + 1)
        for f in files[:5]:
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... ({len(files)} files total)')

## Cell 5 — Collect source pairs

**If Cell 4 shows different folder names than `CXR_png` / `masks`,
update `img_dir` and `mask_dir` below to match.**

In [ ]:
img_dir  = os.path.join(CXRAY_ROOT, 'CXR_png')
mask_dir = os.path.join(CXRAY_ROOT, 'masks')

source_pairs = []
for img_path in sorted(glob.glob(os.path.join(img_dir, '*.png'))):
    fname     = os.path.basename(img_path)
    mask_path = os.path.join(mask_dir, fname)
    if not os.path.exists(mask_path):
        mask_path = os.path.join(mask_dir, fname.replace('.png', '_mask.png'))
    if os.path.exists(mask_path):
        source_pairs.append((img_path, mask_path))

if len(source_pairs) == 0:
    # Fallback: search entire dataset tree for any PNG pairs
    print('WARNING: No pairs found in expected folders.')
    print('Trying auto-discovery...')
    all_imgs = glob.glob(os.path.join(CXRAY_ROOT, '**', '*.png'), recursive=True)
    print(f'All PNGs found: {len(all_imgs)}')
    for p in all_imgs[:10]:
        print(f'  {p}')
    raise RuntimeError('Fix img_dir and mask_dir above based on Cell 4 output.')

VARIANTS_PER_IMAGE = -(-TARGET_SAMPLES // len(source_pairs))
total_expected     = len(source_pairs) * VARIANTS_PER_IMAGE
est_size_gb        = total_expected * 50 / 1e6  # ~50KB per pair

print(f'Source pairs          : {len(source_pairs)}')
print(f'Variants per image    : {VARIANTS_PER_IMAGE}')
print(f'Total expected output : {total_expected:,}')
print(f'Estimated disk usage  : ~{est_size_gb:.1f} GB')
print(f'Available disk space  : {shutil.disk_usage(WORK_DIR).free / 1e9:.1f} GB')

if est_size_gb > shutil.disk_usage(WORK_DIR).free / 1e9 * 0.85:
    print('\nWARNING: Estimated size is close to available space!')
    print('Consider reducing TARGET_SAMPLES in Cell 3.')

## Cell 6 — Enhancement function & corrected pipelines (Albumentations v2 API)

In [ ]:
def enhance_xray(image: np.ndarray) -> np.ndarray:
    gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY) if image.ndim == 3 else image
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    eq    = clahe.apply(gray)
    blur  = cv2.GaussianBlur(eq, (0, 0), sigmaX=2)
    sharp = cv2.addWeighted(eq, 1.5, blur, -0.5, 0)
    return cv2.cvtColor(sharp, cv2.COLOR_GRAY2RGB)


def build_pipelines(img_size):
    
    _R = A.Resize(img_size, img_size)
    BC = cv2.BORDER_CONSTANT

    # ── v2 API helpers (defined once, reused below)
    def GN(lo, hi):   # GaussNoise: convert var_limit to std_range
        import math
        # Calculate standard deviation and normalize to [0.0, 1.0] for v2
        std_lower = math.sqrt(lo) / 255.0
        std_upper = math.sqrt(hi) / 255.0
        return A.GaussNoise(std_range=(std_lower, std_upper), p=1.0)

    def CD(holes, h, w):  # CoarseDropout v2 API
        return A.CoarseDropout(
            num_holes_range=(1, holes),
            hole_height_range=(8, h),
            hole_width_range=(8, w),
            fill=0, p=1.0)

    def DS(lo, hi):   # Downscale v2 API
        return A.Downscale(scale_range=(lo, hi), p=1.0)

    def SZ(val):      # Solarize v2 API
        # Normalize the pixel threshold to a 0.0 - 1.0 range for Albumentations v2
        norm_val = val / 255.0
        return A.Solarize(threshold_range=(norm_val, norm_val), p=1.0)

    return {
        # ── Single transforms ──────────────────────────────────────
        'orig':               A.Compose([_R]),
        'hflip':              A.Compose([_R, A.HorizontalFlip(p=1.0)]),
        'rot_15':             A.Compose([_R, A.Rotate(limit=(15,15),    border_mode=BC, p=1.0)]),
        'rot_neg15':          A.Compose([_R, A.Rotate(limit=(-15,-15),  border_mode=BC, p=1.0)]),
        'rot_30':             A.Compose([_R, A.Rotate(limit=(30,30),    border_mode=BC, p=1.0)]),
        'rot_neg30':          A.Compose([_R, A.Rotate(limit=(-30,-30),  border_mode=BC, p=1.0)]),
        'rot_5':              A.Compose([_R, A.Rotate(limit=(5,5),      border_mode=BC, p=1.0)]),
        'rot_neg5':           A.Compose([_R, A.Rotate(limit=(-5,-5),    border_mode=BC, p=1.0)]),
        'rot_45':             A.Compose([_R, A.Rotate(limit=(45,45),    border_mode=BC, p=1.0)]),
        'rot_neg45':          A.Compose([_R, A.Rotate(limit=(-45,-45),  border_mode=BC, p=1.0)]),
        'elastic_soft':       A.Compose([_R, A.ElasticTransform(alpha=40,  sigma=5,  p=1.0)]),
        'elastic_hard':       A.Compose([_R, A.ElasticTransform(alpha=120, sigma=12, p=1.0)]),
        'elastic_mid':        A.Compose([_R, A.ElasticTransform(alpha=80,  sigma=8,  p=1.0)]),
        'grid_distort':       A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0)]),
        'grid_distort_hard':  A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.5, p=1.0)]),
        'optical_distort':    A.Compose([_R, A.OpticalDistortion(distort_limit=0.3, p=1.0)]),
        'optical_distort_hard':A.Compose([_R, A.OpticalDistortion(distort_limit=0.5, p=1.0)]),
        'bright_up':          A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(0.3,0.3),   contrast_limit=0,           p=1.0)]),
        'bright_down':        A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(-0.3,-0.3), contrast_limit=0,           p=1.0)]),
        'bright_up2':         A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(0.5,0.5),   contrast_limit=0,           p=1.0)]),
        'bright_down2':       A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=(-0.5,-0.5), contrast_limit=0,           p=1.0)]),
        'contrast_up':        A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=0,           contrast_limit=(0.3,0.3),   p=1.0)]),
        'contrast_down':      A.Compose([_R, A.RandomBrightnessContrast(brightness_limit=0,           contrast_limit=(-0.3,-0.3), p=1.0)]),
        'gamma_up':           A.Compose([_R, A.RandomGamma(gamma_limit=(120,150), p=1.0)]),
        'gamma_down':         A.Compose([_R, A.RandomGamma(gamma_limit=(60,80),   p=1.0)]),
        'gamma_mid':          A.Compose([_R, A.RandomGamma(gamma_limit=(90,110),  p=1.0)]),
        'clahe':              A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0)]),
        'clahe2':             A.Compose([_R, A.CLAHE(clip_limit=8.0, p=1.0)]),
        'equalize':           A.Compose([_R, A.Equalize(p=1.0)]),
        'solarize':           A.Compose([_R, SZ(128)]),
        'solarize2':          A.Compose([_R, SZ(64)]),
        'gauss_blur':         A.Compose([_R, A.GaussianBlur(blur_limit=(5,9),   p=1.0)]),
        'gauss_blur2':        A.Compose([_R, A.GaussianBlur(blur_limit=(11,15), p=1.0)]),
        'motion_blur':        A.Compose([_R, A.MotionBlur(blur_limit=9,  p=1.0)]),
        'motion_blur2':       A.Compose([_R, A.MotionBlur(blur_limit=15, p=1.0)]),
        'median_blur':        A.Compose([_R, A.MedianBlur(blur_limit=5,  p=1.0)]),
        'median_blur2':       A.Compose([_R, A.MedianBlur(blur_limit=9,  p=1.0)]),
        'gauss_noise':        A.Compose([_R, GN(20, 80)]),
        'gauss_noise2':       A.Compose([_R, GN(80, 160)]),
        'iso_noise':          A.Compose([_R, A.ISONoise(color_shift=(0.01,0.05), intensity=(0.1,0.4), p=1.0)]),
        'iso_noise2':         A.Compose([_R, A.ISONoise(color_shift=(0.05,0.10), intensity=(0.4,0.8), p=1.0)]),
        'sharpen':            A.Compose([_R, A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'sharpen2':           A.Compose([_R, A.Sharpen(alpha=(0.6,1.0), lightness=(0.8,1.0), p=1.0)]),
        'coarse_dropout':     A.Compose([_R, CD(8,  24, 24)]),
        'coarse_dropout2':    A.Compose([_R, CD(16, 32, 32)]),
        'grid_dropout':       A.Compose([_R, A.GridDropout(ratio=0.3, p=1.0)]),
        'grid_dropout2':      A.Compose([_R, A.GridDropout(ratio=0.5, p=1.0)]),
        'downscale':          A.Compose([_R, DS(0.5,  0.75)]),
        'downscale2':         A.Compose([_R, DS(0.25, 0.5)]),
        'affine':             A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1),  shear=(-10,10), p=1.0)]),
        'affine2':            A.Compose([_R, A.Affine(scale=(0.70,1.30), translate_percent=(-0.15,0.15),shear=(-15,15), p=1.0)]),
        # ── Flip + geometric ──────────────────────────────────────
        'hflip_rot15':        A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(15,15),   border_mode=BC, p=1.0)]),
        'hflip_rot_neg15':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(-15,-15), border_mode=BC, p=1.0)]),
        'hflip_rot30':        A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(30,30),   border_mode=BC, p=1.0)]),
        'hflip_rot_neg30':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=(-30,-30), border_mode=BC, p=1.0)]),
        'hflip_elastic':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60,  sigma=6,  p=1.0)]),
        'hflip_elastic_hard': A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=120, sigma=12, p=1.0)]),
        'hflip_grid':         A.Compose([_R, A.HorizontalFlip(p=1.0), A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0)]),
        'hflip_affine':       A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0)]),
        'hflip_optical':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.OpticalDistortion(distort_limit=0.3, p=1.0)]),
        'hflip_downscale':    A.Compose([_R, A.HorizontalFlip(p=1.0), DS(0.5, 0.75)]),
        # ── Elastic + photometric ──────────────────────────────────
        'elastic_bright':     A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'elastic_bright2':    A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomBrightnessContrast(0.4, 0.4, p=1.0)]),
        'elastic_gamma':      A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'elastic_clahe':      A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'elastic_noise':      A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), GN(20, 80)]),
        'elastic_blur':       A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'elastic_sharpen':    A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'elastic_motion':     A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'elastic_solarize':   A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), SZ(128)]),
        'elastic_equalize':   A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Equalize(p=1.0)]),
        'elastic_dropout':    A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), CD(8, 24, 24)]),
        'elastic_iso':        A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.ISONoise(color_shift=(0.01,0.05), intensity=(0.1,0.4), p=1.0)]),
        # ── Rotate + photometric ───────────────────────────────────
        'rot_bright':         A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot_bright2':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.4, 0.4, p=1.0)]),
        'rot_noise':          A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), GN(10, 40)]),
        'rot_noise2':         A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), GN(40, 100)]),
        'rot_clahe':          A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'rot_gamma':          A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'rot_blur':           A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'rot_motion':         A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'rot_sharpen':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'rot_equalize':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Equalize(p=1.0)]),
        'rot_solarize':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), SZ(128)]),
        'rot_iso':            A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ISONoise(color_shift=(0.01,0.05), intensity=(0.1,0.4), p=1.0)]),
        'rot_dropout':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), CD(8, 24, 24)]),
        'rot_grid_drop':      A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.GridDropout(ratio=0.3, p=1.0)]),
        'rot_elastic':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0)]),
        'rot_affine':         A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0)]),
        'rot_optical':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.OpticalDistortion(distort_limit=0.3, p=1.0)]),
        'rot_downscale':      A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), DS(0.5, 0.75)]),
        # ── Affine + photometric ───────────────────────────────────
        'affine_bright':      A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'affine_noise':       A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), GN(20, 80)]),
        'affine_blur':        A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'affine_clahe':       A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'affine_gamma':       A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'affine_sharpen':     A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'affine_equalize':    A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.Equalize(p=1.0)]),
        'affine_solarize':    A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), SZ(128)]),
        'affine_dropout':     A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), CD(8, 24, 24)]),
        'affine_motion':      A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        # ── Photometric pairs ──────────────────────────────────────
        'bright_clahe':       A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'bright_gamma':       A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'bright_noise':       A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'bright_blur':        A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'bright_sharpen':     A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'bright_equalize':    A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.Equalize(p=1.0)]),
        'bright_motion':      A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'clahe_noise':        A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'clahe_blur':         A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'clahe_sharpen':      A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'clahe_motion':       A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'gamma_noise':        A.Compose([_R, A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'gamma_blur':         A.Compose([_R, A.RandomGamma(gamma_limit=(80,140), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'gamma_sharpen':      A.Compose([_R, A.RandomGamma(gamma_limit=(80,140), p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'noise_blur':         A.Compose([_R, GN(20, 80), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'noise_motion':       A.Compose([_R, GN(20, 80), A.MotionBlur(blur_limit=9, p=1.0)]),
        'noise_sharpen':      A.Compose([_R, GN(20, 80), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'blur_sharpen':       A.Compose([_R, A.GaussianBlur(blur_limit=(3,7), p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'dropout_noise':      A.Compose([_R, CD(8, 24, 24), GN(20, 80)]),
        'dropout_blur':       A.Compose([_R, CD(8, 24, 24), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        # ── Triple combinations ────────────────────────────────────
        'hflip_rot_bright':       A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'hflip_rot_noise':        A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), GN(20, 80)]),
        'hflip_rot_blur':         A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_rot_clahe':        A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'hflip_rot_gamma':        A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'hflip_rot_sharpen':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'hflip_rot_motion':       A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'hflip_rot_equalize':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.Equalize(p=1.0)]),
        'hflip_rot_solarize':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), SZ(128)]),
        'hflip_rot_dropout':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), CD(8, 24, 24)]),
        'hflip_elastic_bright':   A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'hflip_elastic_noise':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), GN(20, 80)]),
        'hflip_elastic_blur':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_elastic_clahe':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'hflip_elastic_gamma':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'hflip_elastic_sharpen':  A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'hflip_elastic_motion':   A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.MotionBlur(blur_limit=9, p=1.0)]),
        'hflip_elastic_dropout':  A.Compose([_R, A.HorizontalFlip(p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), CD(8, 24, 24)]),
        'hflip_affine_bright':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'hflip_affine_noise':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), GN(20, 80)]),
        'hflip_affine_blur':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_affine_clahe':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'hflip_affine_gamma':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'hflip_affine_dropout':   A.Compose([_R, A.HorizontalFlip(p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), CD(8, 24, 24)]),
        'rot_elastic_bright':     A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot_elastic_noise':      A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), GN(20, 80)]),
        'rot_elastic_blur':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'rot_elastic_clahe':      A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'rot_elastic_gamma':      A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'rot_elastic_sharpen':    A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'rot_elastic_dropout':    A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.ElasticTransform(alpha=60, sigma=6, p=1.0), CD(8, 24, 24)]),
        'rot_affine_bright':      A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot_affine_noise':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), GN(20, 80)]),
        'rot_affine_blur':        A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'rot_affine_clahe':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'rot_affine_gamma':       A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'rot_affine_dropout':     A.Compose([_R, A.Rotate(limit=12, border_mode=BC, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), CD(8, 24, 24)]),
        'elastic_affine_bright':  A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'elastic_affine_noise':   A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), GN(20, 80)]),
        'elastic_affine_blur':    A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'elastic_affine_clahe':   A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'elastic_affine_gamma':   A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'elastic_affine_sharpen': A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'elastic_affine_dropout': A.Compose([_R, A.ElasticTransform(alpha=60, sigma=6, p=1.0), A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), CD(8, 24, 24)]),
        'affine_bright_noise':    A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'affine_bright_blur':     A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'affine_bright_clahe':    A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'affine_bright_gamma':    A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'affine_clahe_noise':     A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'affine_gamma_noise':     A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'affine_gamma_blur':      A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'affine_noise_blur':      A.Compose([_R, A.Affine(scale=(0.85,1.15), translate_percent=(-0.1,0.1), shear=(-10,10), p=1.0), GN(20, 80), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'bright_noise_blur':      A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'bright_clahe_noise':     A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'bright_gamma_noise':     A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'bright_gamma_blur':      A.Compose([_R, A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'clahe_gamma_noise':      A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'clahe_gamma_blur':       A.Compose([_R, A.CLAHE(clip_limit=4.0, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'elastic_rot_bright':     A.Compose([_R, A.ElasticTransform(alpha=40, sigma=5, p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'elastic_rot_noise':      A.Compose([_R, A.ElasticTransform(alpha=40, sigma=5, p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), GN(20, 80)]),
        'elastic_rot_clahe':      A.Compose([_R, A.ElasticTransform(alpha=40, sigma=5, p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'elastic_rot_gamma':      A.Compose([_R, A.ElasticTransform(alpha=40, sigma=5, p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'elastic_rot_blur':       A.Compose([_R, A.ElasticTransform(alpha=40, sigma=5, p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'elastic_rot_sharpen':    A.Compose([_R, A.ElasticTransform(alpha=40, sigma=5, p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0)]),
        'elastic_rot_dropout':    A.Compose([_R, A.ElasticTransform(alpha=40, sigma=5, p=1.0), A.Rotate(limit=12, border_mode=BC, p=1.0), CD(8, 24, 24)]),
        'hflip_bright_noise':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'hflip_bright_blur':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_bright_clahe':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0)]),
        'hflip_bright_gamma':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0)]),
        'hflip_clahe_noise':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'hflip_gamma_noise':      A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'hflip_gamma_blur':       A.Compose([_R, A.HorizontalFlip(p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_noise_blur':       A.Compose([_R, A.HorizontalFlip(p=1.0), GN(20, 80), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_dropout_noise':    A.Compose([_R, A.HorizontalFlip(p=1.0), CD(8, 24, 24), GN(20, 80)]),
        'hflip_dropout_blur':     A.Compose([_R, A.HorizontalFlip(p=1.0), CD(8, 24, 24), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'hflip_sharpen_noise':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.Sharpen(alpha=(0.3,0.6), lightness=(0.8,1.0), p=1.0), GN(20, 80)]),
        'hflip_motion_noise':     A.Compose([_R, A.HorizontalFlip(p=1.0), A.MotionBlur(blur_limit=9, p=1.0), GN(20, 80)]),
        'hflip_motion_bright':    A.Compose([_R, A.HorizontalFlip(p=1.0), A.MotionBlur(blur_limit=9, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0)]),
        'rot30_bright_noise':     A.Compose([_R, A.Rotate(limit=(30,30), border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'rot30_bright_blur':      A.Compose([_R, A.Rotate(limit=(30,30), border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
        'rot30_clahe_noise':      A.Compose([_R, A.Rotate(limit=(30,30), border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'rot30_gamma_noise':      A.Compose([_R, A.Rotate(limit=(30,30), border_mode=BC, p=1.0), A.RandomGamma(gamma_limit=(80,140), p=1.0), GN(20, 80)]),
        'rot_neg30_bright_noise': A.Compose([_R, A.Rotate(limit=(-30,-30), border_mode=BC, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'rot_neg30_clahe_noise':  A.Compose([_R, A.Rotate(limit=(-30,-30), border_mode=BC, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'grid_bright_noise':      A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'grid_clahe_noise':       A.Compose([_R, A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), GN(20, 80)]),
        'optical_bright_noise':   A.Compose([_R, A.OpticalDistortion(distort_limit=0.3, p=1.0), A.RandomBrightnessContrast(0.2, 0.2, p=1.0), GN(20, 80)]),
        'optical_clahe_blur':     A.Compose([_R, A.OpticalDistortion(distort_limit=0.3, p=1.0), A.CLAHE(clip_limit=4.0, p=1.0), A.GaussianBlur(blur_limit=(3,7), p=1.0)]),
    }


import warnings
with warnings.catch_warnings():
    warnings.simplefilter('error') 
    try:
        test_p = build_pipelines(IMG_SIZE)
        print(f'Pipelines built cleanly — {len(test_p)} total, {VARIANTS_PER_IMAGE} needed')
        assert len(test_p) >= VARIANTS_PER_IMAGE
        del test_p
        print('No API warnings — all transforms are v2 compatible.')
    except Warning as w:
        print(f'WARNING still present: {w}')

## Cell 7 — Parallel augmentation with disk space monitoring

Progress is printed every 10 source images.
The job will stop early and save a checkpoint if disk space drops below 1 GB.

In [ ]:
def worker_augment(args):
    """
    Worker: processes one source image through all pipeline variants.
    Builds pipelines internally to avoid cross-process pickling issues.
    Returns (count_written, base_name).
    """
    img_path, mask_path, aug_img_dir, aug_mask_dir, img_size, n_variants = args

    import cv2, numpy as np, albumentations as A
    from PIL import Image

    pipelines      = build_pipelines(img_size)
    pipeline_items = list(pipelines.items())[:n_variants]

    img  = enhance_xray(np.array(Image.open(img_path).convert('RGB')))
    mask = np.array(Image.open(mask_path).convert('L'))
    base = os.path.splitext(os.path.basename(img_path))[0]

    count = 0
    for suffix, pipeline in pipeline_items:
        out = pipeline(image=img, mask=mask)
        Image.fromarray(out['image']).save(
            os.path.join(aug_img_dir, f'{base}_{suffix}.png'))
        Image.fromarray(out['mask']).save(
            os.path.join(aug_mask_dir, f'{base}_{suffix}_mask.png'))
        count += 1
    return count, base


worker_args = [
    (img_p, msk_p, AUG_IMG_DIR, AUG_MASK_DIR, IMG_SIZE, VARIANTS_PER_IMAGE)
    for img_p, msk_p in source_pairs
]

print(f'Starting augmentation: {len(source_pairs)} images × {VARIANTS_PER_IMAGE} variants')
print(f'Workers: {N_WORKERS}')
print(f'─' * 65)

t0         = time.time()
total      = 0
done_imgs  = 0
stopped_early = False

with Pool(processes=N_WORKERS) as pool:
    for count, base in pool.imap_unordered(worker_augment, worker_args):
        total     += count
        done_imgs += 1

        if done_imgs % 10 == 0 or done_imgs == len(source_pairs):
            elapsed  = time.time() - t0
            eta      = elapsed / done_imgs * (len(source_pairs) - done_imgs)
            free_gb  = shutil.disk_usage(WORK_DIR).free / 1e9
            used_gb  = shutil.disk_usage(WORK_DIR).used / 1e9
            print(f'  [{done_imgs:>3}/{len(source_pairs)}] '
                  f'{total:>7,} files | '
                  f'disk used {used_gb:.1f} GB / free {free_gb:.1f} GB | '
                  f'elapsed {elapsed/60:.1f}m | '
                  f'ETA {eta/60:.1f}m')

            # Safety stop if disk is nearly full
            if free_gb < 1.0:
                print(f'\n  STOPPING EARLY — only {free_gb:.2f} GB free!')
                print(f'  Completed {done_imgs}/{len(source_pairs)} source images.')
                pool.terminate()
                stopped_early = True
                break

elapsed_total = time.time() - t0
n_imgs  = len(glob.glob(os.path.join(AUG_IMG_DIR,  '*.png')))
n_masks = len(glob.glob(os.path.join(AUG_MASK_DIR, '*.png')))

print(f'\n{"─"*65}')
print(f'Augmentation {"STOPPED EARLY" if stopped_early else "complete"}')
print(f'  Time elapsed    : {elapsed_total/60:.1f} min')
print(f'  Images on disk  : {n_imgs:,}')
print(f'  Masks on disk   : {n_masks:,}')
print(f'  Disk used       : {shutil.disk_usage(WORK_DIR).used / 1e9:.1f} GB')
print(f'  Disk free       : {shutil.disk_usage(WORK_DIR).free / 1e9:.1f} GB')

if n_imgs != n_masks:
    print(f'  WARNING: image/mask count mismatch ({n_imgs} vs {n_masks})!')
else:
    print(f'  Image/mask counts match ✓')

# Save a manifest so the training notebook knows what it received
manifest = {
    'total_pairs':         n_imgs,
    'source_images':       done_imgs,
    'variants_per_image':  VARIANTS_PER_IMAGE,
    'img_size':            IMG_SIZE,
    'stopped_early':       stopped_early,
    'elapsed_minutes':     round(elapsed_total / 60, 1),
}
with open(os.path.join(WORK_DIR, 'aug_manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'\nManifest saved to /kaggle/working/aug_manifest.json')
print(json.dumps(manifest, indent=2))

## Cell 8 — Verify a few samples visually

In [ ]:
import matplotlib.pyplot as plt, random

all_masks = sorted(glob.glob(os.path.join(AUG_MASK_DIR, '*.png')))
sample_masks = random.sample(all_masks, min(4, len(all_masks)))

fig, axes = plt.subplots(len(sample_masks), 2, figsize=(6, 3 * len(sample_masks)))
fig.suptitle('Augmentation samples (image | mask)', fontweight='bold')

for row, mask_path in enumerate(sample_masks):
    img_name = os.path.basename(mask_path).replace('_mask', '')
    img_path = os.path.join(AUG_IMG_DIR, img_name)
    axes[row, 0].imshow(np.array(Image.open(img_path)))
    axes[row, 0].set_title(img_name[:30], fontsize=7)
    axes[row, 0].axis('off')
    axes[row, 1].imshow(np.array(Image.open(mask_path)), cmap='gray')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'aug_samples.png'), dpi=100)
plt.show()
print('Sample verification complete.')

## Cell 9 — Done. Publish as Kaggle Dataset.

```
Output files in /kaggle/working/:
  aug_cxray/
    images/   ← augmented X-ray PNGs
    masks/    ← augmented mask PNGs
  aug_manifest.json
  aug_samples.png
```

**To publish:**
1. In the Kaggle notebook sidebar → **Output** → find `/kaggle/working/`
2. Click **"New Dataset"**
3. Name it (e.g. `lung-seg-augmented`) — it defaults to **Private**
4. Click **Create**
5. Copy your `username/dataset-slug` for use in the Colab training notebook
   at: `KAGGLE_DATASET = 'your_username/lung-seg-augmented'`

In [ ]:
import os
import json
import subprocess
import shutil
from kaggle_secrets import UserSecretsClient

# 1. Retrieve credentials
try:
    user_secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_KEY")
except Exception as e:
    print("Error: Make sure you have attached KAGGLE_USERNAME and KAGGLE_KEY in Add-ons -> Secrets")
    raise e

DATASET_TITLE = "My Augmented X-Ray Variants"
DATASET_SLUG  = "augmented-xray-variants-v1" 
USERNAME = os.environ['KAGGLE_USERNAME']
OUTPUT_DIR = "/kaggle/working" 

# 2. MANUALLY ZIP THE FOLDER (The Bulletproof Fix)
folder_to_zip = "/kaggle/working/aug_cxray"
zip_destination = "/kaggle/working/aug_cxray_data" # it will automatically add .zip

print(f"Zipping {folder_to_zip}... This might take a few minutes for 40k images.")
shutil.make_archive(zip_destination, 'zip', folder_to_zip)
print(f"Successfully created: {zip_destination}.zip")

# 3. Create the dataset-metadata.json file
metadata = {
  "title": DATASET_TITLE,
  "id": f"{USERNAME}/{DATASET_SLUG}",
  "licenses": [{"name": "CC0-1.0"}] 
}

metadata_path = os.path.join(OUTPUT_DIR, "dataset-metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Metadata generated for {USERNAME}/{DATASET_SLUG}")

# 4. Push to Kaggle
print("Uploading dataset to Kaggle...")

# We use subprocess list formatting which is much more stable than Jupyter ! commands
# We are pushing the version update directly since the dataset shell already exists
result = subprocess.run(
    ["kaggle", "datasets", "version", "-p", OUTPUT_DIR, "-m", "Uploading zipped images"],
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print("Errors/Warnings:", result.stderr)

print(" Upload triggered! Check your dataset page in a few minutes.")